# DumpWatch 2.0 Task B · Frozen Real Only pipeline

This notebook freezes and verifies the original Real Only SVM using the audited training and development pairs. The final test cell defaults to disabled. Running all cells in this shipped state does not open reserved test JPEG members. Reported development scores are exploratory.

The pair inventory is embedded. Drive supplies the previously uploaded verified real ZIP and unchanged split CSV. The notebook writes artifacts to a timestamped runtime folder. No GPU is needed.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get("DUMPWATCH_PACKAGE", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "project_setup.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within the unpacked code folder or set DUMPWATCH_PACKAGE to it. See SETUP.md.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from project_setup import project_paths, new_output, enable_demo_vendor
P = project_paths()
INPUTS = P["manifest_root"]
REAL_ZIP = P["real_zip"]
SPLIT_CSV = INPUTS / "real_location_split.csv"
PAIR_CSV = INPUTS / "final_pair_inventory.csv"
OUTPUT_ROOT = new_output("selected_svm_training")
FINAL_TEST_APPROVED = False
APPROVED_FROZEN_MODEL = None
APPROVED_FROZEN_MODEL_SHA256 = ""
APPROVED_TEST_PAIRS = None
APPROVED_TEST_PAIRS_SHA256 = ""
print("Recovered ORIGINAL fitting code. Following cells TRAIN a new model into:", OUTPUT_ROOT)
print("The selected submitted model is never overwritten. Reserved test remains disabled.")


## Training provenance and fixed preprocessing

Use the visually reviewed ground crops, resized to 128 × 128 with bilinear interpolation. The original descriptor computes 9 unsigned gradient bins and 6 RGB moments per cell on a 4 × 4 grid. W, A, their absolute descriptor differences, and three appearance-margin terms give 723 features. Standardize using training data only. The two linear SVMs use class balance and fixed C = 0.1. Out-of-location appearance scores feed pair training and repeated outcomes share case weight. All 20 reviewed training pairs are used. Development is a replay check.


In [ ]:
import csv, hashlib, io, json, zipfile
from collections import Counter, defaultdict
import numpy as np
from PIL import Image

TRAIN = {f"location_{n:02}" for n in (1,3,5,6,7,9)}
DEV = {"location_02", "location_08"}
TEST = {"location_04", "location_10"}
REAL_HASH = "1e97cfe51b4ef7fc9e61aaa7cbd2cbce4ac5a24fd274f07503d7deb5138f0412"
SPLIT_HASH = "0ae3ff815747ba5c8c677fb3670c373AECBE9ECF2B4E3BD6B2410450748A3E60".lower()

def sha(data):
    return hashlib.sha256(data).hexdigest()

def require(ok, message):
    if not ok:
        raise ValueError(message)

def box_from_string(value):
    return [float(n) for n in value.split()]

def descriptor(raw, box):
    left,top,right,bottom=box
    require(0<=left<right<=1 and 0<=top<bottom<=1, f"Invalid normalized ROI {box}")
    with Image.open(io.BytesIO(raw)) as source:
        w,h=source.size
        pixels=source.convert("RGB").crop((round(left*w),round(top*h),round(right*w),round(bottom*h)))
    require(min(pixels.size)>=64, "ROI too small to assess")
    rgb=np.asarray(pixels.resize((128,128),Image.Resampling.BILINEAR),dtype=np.float32)/255
    gray=rgb @ np.array([.299,.587,.114],dtype=np.float32)
    dy,dx=np.gradient(gray)
    magnitude=np.hypot(dx,dy)
    orientation=np.floor((np.arctan2(dy,dx)%np.pi)*9/np.pi).astype(int).clip(0,8)
    features=[]
    for y in range(0,128,32):
        for x in range(0,128,32):
            region=np.s_[y:y+32,x:x+32]
            hist=np.bincount(orientation[region].ravel(),weights=magnitude[region].ravel(),minlength=9)
            features.extend(hist/(np.linalg.norm(hist)+1e-6))
            patch=rgb[region]
            features.extend(patch.mean(axis=(0,1)))
            features.extend(patch.std(axis=(0,1)))
    return np.asarray(features,dtype=np.float32)

def paired_vector(w,a,scores):
    return np.r_[w,a,np.abs(w-a),scores[0],scores[1],scores[0]-scores[1]]

def select_clearance(score, visually_assessable, low, high):
    require(np.isfinite(low) and np.isfinite(high) and low<high,"Invalid abstention interval")
    return np.where(~np.asarray(visually_assessable,dtype=bool),"abstain",
           np.where(score>=high,"fully_cleared",np.where(score<=low,"waste_still_present","abstain")))

def evaluate(target,prediction):
    target,prediction=np.asarray(target),np.asarray(prediction)
    remaining=target==0
    cleared=target==1
    false=int((remaining & (prediction=="fully_cleared")).sum())
    valid=int((prediction!="abstain").sum())
    return dict(false_clearance=false,remaining_total=int(remaining.sum()),
        false_clearance_rate=false/remaining.sum() if remaining.any() else None,
        correctly_cleared=int((cleared & (prediction=="fully_cleared")).sum()),
        cleared_total=int(cleared.sum()),cleared_as_waste_present=int((cleared & (prediction=="waste_still_present")).sum()),
        waste_present_correct=int((remaining & (prediction=="waste_still_present")).sum()),
        evaluated=len(target),coverage=valid/len(target) if len(target) else None,
        abstentions=int(len(target)-valid))

In [ ]:
def real_features():
    require(sha(REAL_ZIP.read_bytes())==REAL_HASH,"Real ZIP is not the frozen archive")
    require(sha(SPLIT_CSV.read_bytes())==SPLIT_HASH,"Frozen split CSV mismatch")
    with SPLIT_CSV.open(newline="",encoding="utf-8-sig") as f:
        mapping={x["location_id"]:x["split"] for x in csv.DictReader(f)}
    require(mapping=={**dict.fromkeys(TRAIN,"train"),**dict.fromkeys(DEV,"validation"),**dict.fromkeys(TEST,"test")},
            "Unexpected split membership")
    with PAIR_CSV.open(newline="",encoding="utf-8-sig") as f:
        review=list(csv.DictReader(f))
    require(len(review)==48 and len({r["pair_id"] for r in review})==48,"Unexpected pair inventory")
    require({r["location_id"] for r in review}==TRAIN|DEV,"Unexpected or reserved pair location")
    selected=[r for r in review if r["status"]=="usable"]
    seen=set()
    result=[]
    with zipfile.ZipFile(REAL_ZIP) as z:
        archived={r["relative_path"]:r for r in csv.DictReader(io.StringIO(z.read("dataset_manifest.csv").decode("utf-8-sig")))}
        for row in selected:
            loc=row["location_id"]
            split="training" if loc in TRAIN else "development"
            require(row["split"]==split,f"Incorrect split for {row['pair_id']}")
            require(row["inspection_sha256"] not in seen,"Duplicate eligible inspection hash")
            seen.add(row["inspection_sha256"])
            role=row["inspection_stage"]
            require(role in {"A_partial","A_removed"},"Dependent unchanged controls are excluded")
            target=int(role=="A_removed")
            require(row["target"]==("fully_cleared" if target else "waste_still_present"),"Target mismatch")
            vectors=[]
            for column,hash_column,roi_column,stage in (
                    ("W_path","W_sha256","W_roi_xyxy","W"),
                    ("inspection_path","inspection_sha256","inspection_roi_xyxy",role)):
                path=row[column]
                frozen=archived[path]
                # Enforce allowed location and same case before opening any member.
                require(path.startswith(f"real/{loc}/") and frozen["location_id"]==loc and loc not in TEST
                        and frozen["case_id"]==row["case_id"] and frozen["semantic_stage"]==stage,
                        f"Unexpected member: {path}")
                raw=z.read(path)
                require(sha(raw)==row[hash_column]==frozen["file_hash"],f"Image hash mismatch: {path}")
                vectors.append(descriptor(raw,box_from_string(row[roi_column])))
            require(row["W_sha256"]!=row["inspection_sha256"],"Exact W control in real index")
            result.append(dict(row,w=vectors[0],a=vectors[1],y=target))
    hash_membership=defaultdict(set)
    for row in result:
        for field in ("W_sha256","inspection_sha256"):
            hash_membership[row[field]].add(row["split"])
    require(all(len(v)==1 for v in hash_membership.values()),"A real image hash crosses splits")
    return result,review

real,review=real_features()
print("Reviewed pairs:",dict(Counter((r["split"],r["target"]) for r in real)))
print("No V1 or reserved test images were used in model preparation.")


In [ ]:
def appearance_records(rows):
    unique={}
    for row in rows:
        for part,feature,y in (("W_sha256",row["w"],1),
                               ("inspection_sha256",row["a"],1-row["y"])):
            key=row[part]
            current=unique.get(key)
            require(current is None or current["y"]==y,"Conflicting appearance labels")
            unique.setdefault(key,dict(x=feature,y=y,site=row["location_id"],sha256=key))
    return list(unique.values())

def make_svm():
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import LinearSVC
    return make_pipeline(StandardScaler(),LinearSVC(C=.1,class_weight="balanced",random_state=37,max_iter=20000))

def appearance_model(rows, synthetic):
    records=appearance_records(rows)
    train=records+synthetic
    estimator=make_svm()
    # Equal treatment across arms, with capped auxiliary mass in the assisted arm.
    weight=np.r_[np.ones(len(records)),np.full(len(synthetic),min(.25,len(records)/(4*max(1,len(synthetic)))))]
    x=np.stack([r["x"] for r in train]); y=np.array([r["y"] for r in train])
    estimator.fit(x,y,standardscaler__sample_weight=weight,linearsvc__sample_weight=weight)
    return estimator

def vector_for(model,row):
    scores=model.decision_function(np.stack((row["w"],row["a"])))
    return paired_vector(row["w"],row["a"],scores)


In [ ]:
import csv,hashlib,io,json,re,zipfile
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter
import numpy as np
from PIL import Image

def frozen_standardized_linear(estimator):
    scaler=estimator.named_steps["standardscaler"]
    svm=estimator.named_steps["linearsvc"]
    return dict(mean=scaler.mean_.tolist(),scale=scaler.scale_.tolist(),
                coefficients=svm.coef_[0].tolist(),intercept=float(svm.intercept_[0]),
                class_order=svm.classes_.tolist())

def frozen_score(values,spec):
    values=np.array(values,copy=True)
    # Match StandardScaler's in-place float32 rounding for appearance inputs.
    values-=np.asarray(spec["mean"])
    values/=np.asarray(spec["scale"])
    return values @ np.asarray(spec["coefficients"])+spec["intercept"]

def frozen_pair_features(w,a,model):
    scores=frozen_score(np.stack((w,a)),model["appearance"])
    return paired_vector(w,a,scores)

def fit_frozen_baseline(train,development):
    from threadpoolctl import threadpool_limits
    from importlib.metadata import version
    sites=sorted({r["location_id"] for r in train})
    case_counts=Counter((r["location_id"],r["case_id"]) for r in train)
    oof=[]
    with threadpool_limits(limits=1):
        for site in sites:
            inner=[r for r in train if r["location_id"]!=site]
            held=[r for r in train if r["location_id"]==site]
            aux=appearance_model(inner,[])
            for r in held:
                oof.append((vector_for(aux,r),r["y"],1/case_counts[r["location_id"],r["case_id"]]))
        pair=make_svm()
        x,y,w=zip(*oof)
        pair.fit(np.stack(x),np.asarray(y),standardscaler__sample_weight=w,linearsvc__sample_weight=w)
        aux=appearance_model(train,[])
    frozen=dict(version="task-b-frozen-real-only-v1",
                archive_sha256=REAL_HASH,split_sha256=SPLIT_HASH,pair_config_sha256=sha(PAIR_CSV.read_bytes()),
                feature_spec="ROI RGB 128x128, 4x4 bins of 9 unsigned orientations and 6 RGB moments; W,A,abs difference,3 appearance scores",
                hyperparameters=dict(C=.1,class_weight="balanced",random_state=37,max_iter=20000,
                                     duplicate_policy="case_weight",appearance_fold="leave_real_location_out"),
                policy=dict(low=-.5,high=.5,unassessable="abstain",forced_diagnostic_threshold=0),
                packages={p:version(p) for p in ("numpy","Pillow","scikit-learn","scipy")},
                training_pairs=[r["pair_id"] for r in train],
                appearance=frozen_standardized_linear(aux),pair=frozen_standardized_linear(pair))
    X=np.stack([vector_for(aux,r) for r in development])
    scores=pair.decision_function(X)
    replay=frozen_score(np.stack([frozen_pair_features(r["w"],r["a"],frozen) for r in development]),frozen["pair"])
    require(np.allclose(scores,replay,atol=1e-8,rtol=0),"Frozen model replay mismatch")
    frozen["development_replay_max_abs_margin_delta"]=float(np.max(np.abs(scores-replay)))
    return frozen,replay

train=[r for r in real if r["split"]=="training"]
dev=[r for r in real if r["split"]=="development"]
require(Counter(r["y"] for r in train)=={0:14,1:6} and Counter(r["y"] for r in dev)=={0:6,1:2},"Frozen pair count mismatch")
frozen,margins=fit_frozen_baseline(train,dev)
policy=frozen["policy"]
fixed=select_clearance(margins,np.ones(len(dev),bool),policy["low"],policy["high"])
forced=select_clearance(margins,np.ones(len(dev),bool),-1e-12,1e-12)
run=OUTPUT_ROOT / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run.mkdir(parents=True,exist_ok=False)
(run/"frozen_real_only_svm.json").write_text(json.dumps(frozen,indent=2)+"\n",encoding="utf-8")
rows=[dict(pair_id=r["pair_id"],location=r["location_id"],truth=r["target"],score=float(score),
           fixed_prediction=decision,forced_diagnostic_prediction=zero)
      for r,score,decision,zero in zip(dev,margins,fixed,forced)]
with (run/"development_replay.csv").open("x",newline="",encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
y=np.asarray([r["y"] for r in dev])
print("Development-only fixed rule:",evaluate(y,fixed))
print("Development-only forced diagnostic:",evaluate(y,forced))
print("Frozen model SHA-256:",sha((run/"frozen_real_only_svm.json").read_bytes()))
print("Frozen files:",run)


## Final evaluation gate

Reserved locations are 04 and 10. Before enabling final evaluation, obtain separate authorization to open the reserved images and create a blinded pair inventory. Assign inspections by recorded provenance and visual coverage before generating model predictions. Lock the inventory and frozen model by SHA-256. Have a reviewer confirm the case links and use each inspection hash at most once. A known but unassessable pair is included as an abstention, with the reason. Missing or unconfirmed outcomes are documented as excluded observations outside the scoring denominator. The template has headers only.

Use the shipped fixed policy. Margins at or below -0.5 predict waste still present. Margins at or above +0.5 predict fully cleared. Other margins or unassessable pairs abstain. The zero threshold is a separately labeled diagnostic. Do not tune either policy on the reserved set.


In [ ]:
def read_approved_test_pairs(path):
    with path.open(encoding="utf-8-sig",newline="") as f:
        rows=list(csv.DictReader(f))
    require(rows and {r["location_id"] for r in rows}==TEST,"Approved manifest must cover both reserved locations")
    require(len({r["pair_id"] for r in rows})==len(rows),"Duplicate pair IDs")
    require(len({(r["location_id"],r["case_id"],r["inspection_stage"]) for r in rows})==len(rows),
            "At most one row per real case and outcome")
    return rows

def score_authorized_test(path,model):
    rows=read_approved_test_pairs(path)
    output=[]
    seen=set()
    with zipfile.ZipFile(REAL_ZIP) as archive:
        index={r["relative_path"]:r for r in csv.DictReader(io.StringIO(archive.read("dataset_manifest.csv").decode("utf-8-sig")))}
        for r in rows:
            loc,case,role=r["location_id"],r["case_id"],r["inspection_stage"]
            require(loc in TEST and role in {"A_partial","A_removed"},"Unsupported test location or label")
            w_path,a_path=r["W_path"],r["inspection_path"]
            w_meta,a_meta=index[w_path],index[a_path]
            for member,row,expected in ((w_path,w_meta,"W"),(a_path,a_meta,role)):
                require(member.startswith(f"real/{loc}/{case}/") and row["location_id"]==loc
                        and row["case_id"]==case and row["semantic_stage"]==expected,
                        f"Test manifest mismatch: {member}")
            h=a_meta["file_hash"]
            require(h!=w_meta["file_hash"] and h not in seen,"Dependent or repeated test inspection")
            require({x["location_id"] for x in index.values() if x["file_hash"]==h}=={loc},
                    "Inspection hash crosses locations")
            seen.add(h)
            status=r["assessment_status"]
            require(status in {"usable","unassessable"},"Missing visual coverage decision")
            y=int(role=="A_removed")
            value=None
            if status=="usable":
                vectors=[]
                for member,meta,key in ((w_path,w_meta,"W_roi_xyxy"),(a_path,a_meta,"inspection_roi_xyxy")):
                    raw=archive.read(member)
                    require(sha(raw)==meta["file_hash"],"Test member hash mismatch")
                    vectors.append(descriptor(raw,box_from_string(r[key])))
                value=float(frozen_score(frozen_pair_features(*vectors,model),model["pair"]))
            prediction=select_clearance(np.asarray([value or 0]),[status=="usable"],
                                         model["policy"]["low"],model["policy"]["high"])[0]
            forced="abstain" if value is None else "fully_cleared" if value>=0 else "waste_still_present"
            output.append(dict(pair_id=r["pair_id"],location_id=loc,truth=y,score=value,
              prediction=prediction,forced_diagnostic_prediction=forced,assessment_status=status,
              reason=r["review_reason"] if status=="unassessable" else "inside fixed score interval" if prediction=="abstain" else ""))
    return output

if FINAL_TEST_APPROVED:
    require(APPROVED_TEST_PAIRS is not None and APPROVED_FROZEN_MODEL is not None,
            "Provide a finalized test pair manifest and the previously frozen model")
    require(re.fullmatch("[0-9a-f]{64}",APPROVED_TEST_PAIRS_SHA256)
            and re.fullmatch("[0-9a-f]{64}",APPROVED_FROZEN_MODEL_SHA256),"Pin both approved hashes before test")
    raw=Path(APPROVED_FROZEN_MODEL).read_bytes()
    require(sha(raw)==APPROVED_FROZEN_MODEL_SHA256,"Frozen model hash mismatch")
    require(sha(Path(APPROVED_TEST_PAIRS).read_bytes())==APPROVED_TEST_PAIRS_SHA256,"Test pair inventory hash mismatch")
    test_model=json.loads(raw)
    require(test_model["version"]=="task-b-frozen-real-only-v1"
            and (test_model["archive_sha256"],test_model["split_sha256"],test_model["pair_config_sha256"])
                ==(REAL_HASH,SPLIT_HASH,sha(PAIR_CSV.read_bytes()))
            and test_model["policy"]=={"low":-.5,"high":.5,"unassessable":"abstain","forced_diagnostic_threshold":0},
            "Frozen model policy or provenance mismatch")
    test_rows=score_authorized_test(Path(APPROVED_TEST_PAIRS),test_model)
    path=run/"reserved_test_predictions.csv"
    with path.open("x",newline="",encoding="utf-8") as f:
        writer=csv.DictWriter(f,fieldnames=list(test_rows[0]));writer.writeheader();writer.writerows(test_rows)
    for scope,subset in [("all",test_rows),*((loc,[r for r in test_rows if r["location_id"]==loc]) for loc in sorted(TEST))]:
        print(scope,"fixed",evaluate([r["truth"] for r in subset],[r["prediction"] for r in subset]))
        print(scope,"forced diagnostic",evaluate([r["truth"] for r in subset],[r["forced_diagnostic_prediction"] for r in subset]))
else:
    print("Reserved test disabled. No test JPEG member was opened.")
